[![Mở trong Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trungnguyenvn/mlsysbook-vi-notebooks/blob/main/vol1-ch06-nn-architectures/04-tu-khoi-dung-chung-toi-quyet-dinh.ipynb)

# Từ khối dùng chung tới một quyết định kiến trúc

Ba sổ tay trước đếm từng họ kiến trúc riêng rẽ. Sổ tay này đi theo nửa sau của chương: những khối mà mọi họ
dùng chung, cái giá năng lượng của việc di chuyển dữ liệu, họ kiến trúc thứ năm (hệ gợi ý, RecSys) nơi giới hạn
là dung lượng chứ không phải phép tính, và cuối cùng là cách biến tất cả những con số đó thành một quyết định.

**Bạn sẽ làm:**
1. so chuẩn hóa theo batch (batch normalization) với chuẩn hóa theo tầng (layer normalization): mẫu này có phụ
   thuộc mẫu kia không, huấn luyện và suy luận có cho cùng kết quả không;
2. kiểm Định lý 1.2 về kết nối bỏ qua bằng một phép tính ngắn;
3. tính năng lượng của một tầng dày đặc và một tầng tích chập theo hai con số của Horowitz;
4. tính bức tường dung lượng của bảng embedding, và đo một lần tra bảng trên CPU này;
5. dựng bảng độ phức tạp cho bốn họ kiến trúc, rồi đặt ba chữ ký tải công việc vào định luật sắt (iron law);
6. chọn kiến trúc cho máy bẫy ảnh động vật hoang dã của chương bằng một bảng khả thi tính được.

In [ ]:
#@title Chuẩn bị: thư viện, hạt giống và hai hàm đối chiếu với sách { display-mode: "form" }
# Chạy ô này trước. Nó không cài thêm gì: mọi thư viện dùng ở đây đều có sẵn trên Colab.
import math, os, random, re, sys, time

import matplotlib.pyplot as plt
import numpy as np
import torch

# Số luồng CPU. 0 để thư viện tự chọn (trên Colab là 2).
NB_THREADS = int(os.environ.get("NB_THREADS", "0"))
if NB_THREADS:
    torch.set_num_threads(NB_THREADS)
    try:
        torch.set_num_interop_threads(NB_THREADS)
    except RuntimeError:  # chỉ đặt được một lần mỗi phiên
        pass
    from threadpoolctl import threadpool_limits
    threadpool_limits(NB_THREADS)

SEED = 42


def dat_hat_giong(seed=SEED):
    """Cùng hạt giống thì cùng kết quả, mỗi lần chạy lại."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


dat_hat_giong()
DATA = os.environ.get("NB_DATA", "data")  # nơi tải dữ liệu về
NB_STRICT = os.environ.get("NB_STRICT") == "1"
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.3})

# Đọc một con số theo cách bảng số liệu của bài học viết nó (dấu chấm hay dấu phẩy).
import math
import re

_SUP = str.maketrans("⁰¹²³⁴⁵⁶⁷⁸⁹⁻⁺", "0123456789-+")
_MULT = {"tỷ": 1e9, "tỉ": 1e9, "triệu": 1e6, "nghìn": 1e3, "ngàn": 1e3,
         "K": 1e3, "M": 1e6, "B": 1e9}
_NUM = re.compile(
    r"(?<![\w.,])(-?)(\d+(?:[.,]\d+)*)"                      # sign, digits with separators
    r"(?:\s*[×x·]\s*10\s*(?:\^\{?\s*([-+]?\d+)\s*\}?|([⁻⁺]?[⁰¹²³⁴⁵⁶⁷⁸⁹]+))|[eE]([-+]?\d+)"
    r"|\^\{?\s*([-+]?\d+)\s*\}?|([⁻⁺]?[⁰¹²³⁴⁵⁶⁷⁸⁹]+))?"                  # or a bare power, 10^{-5}
    r"(\s*%|\s*(?:tỷ|tỉ|triệu|nghìn|ngàn)(?!\w)|\s?[KMB](?![A-Za-z]))?")


def norm(text):
    """The outline's markup and typography reduced to plain text, as both sides compare it."""
    text = text.replace("{,}", ",").replace("**", "").replace("`", "")
    text = re.sub(r"[   ]", " ", text)
    text = re.sub(r"−(?=\d)", "-", text)
    text = re.sub(r"[~≈]", "", text)
    return re.sub(r"\s+", " ", text).strip()


def _readings(digits):
    """(value, decimals) for each way the separators in `digits` can be read."""
    out = []
    groups_ok = lambda parts: all(len(p) == 3 for p in parts[1:])
    for thou, dec in ((",", "."), (".", ",")):
        if thou in digits and dec in digits:
            head, _, tail = digits.rpartition(dec)
            if dec in head or not tail.isdigit() or not groups_ok(head.split(thou)):
                continue
            out.append((float(head.replace(thou, "") + "." + tail), len(tail)))
    for sep in (",", "."):
        other = "." if sep == "," else ","
        if sep in digits and other not in digits:
            parts = digits.split(sep)
            if groups_ok(parts):
                out.append((float("".join(parts)), 0))                    # thousands
            if len(parts) == 2:
                out.append((float(parts[0] + "." + parts[1]), len(parts[1])))  # decimal
    if "." not in digits and "," not in digits:
        out.append((float(digits), 0))
    return out


def candidates(text):
    """Every (value, half_unit) the numbers in `text` can mean.

    half_unit is half of the last printed digit, scaled like the value, so `437,5 KB`
    accepts 437.544. Each number is offered as written and, when a multiplier or `%`
    follows it, scaled by that too, so `8,5 tỉ` cites 8.5 or 8.5e9 and `20 %` cites 20 or 0.2.
    """
    out = []
    for m in _NUM.finditer(norm(text)):
        sign, digits, exp_caret, exp_sup, exp_e, pow_caret, pow_sup, suffix = m.groups()
        exp = exp_caret or (exp_sup.translate(_SUP) if exp_sup else None) or exp_e
        power = pow_caret or (pow_sup.translate(_SUP) if pow_sup else None)
        scale10 = 10.0 ** int(exp) if exp else 1.0
        suffix = (suffix or "").strip()
        extra = 0.01 if suffix == "%" else _MULT.get(suffix)
        for value, decimals in _readings(digits):
            if power:
                # 10^{-5}, 2³²: an exact value, so it is matched to 1 part in 10⁹; a looser
                # comparison needs an explicit tol.
                v = value ** int(power)
                half = abs(v) * 1e-9
            else:
                v = value * scale10
                half = 0.5 * 10.0 ** -decimals * scale10
            v = -v if sign else v
            out.append((v, half))
            if extra:
                out.append((v * extra, half * extra))
    return out


def match(value, text):
    """The (value, half_unit) reading of `text` that `value` equals, or None."""
    for v, half in candidates(text):
        if math.isclose(value, v, rel_tol=1e-9, abs_tol=1e-300):
            return v, half
    return None


def well_anchored(text):
    """False for a citation too bare to identify one cell, like "1" or "2"."""
    t = norm(text)
    rest = re.sub(r"[\d.,\s\-+]", "", t)
    if rest:
        return True
    sig = re.sub(r"[^\d]", "", t).lstrip("0")
    return len(sig) >= 2


def _so(x):
    """Một con số để in: số nguyên có dấu phẩy ngăn nghìn, số thực giữ 6 chữ số có nghĩa."""
    if isinstance(x, (int, np.integer)) or (isinstance(x, float) and x.is_integer() and abs(x) < 1e15):
        return f"{int(x):,}"
    return f"{x:,.6g}"


def sach(gia_tri, nguon=None, trich=None, tol=None):
    """Một con số của sách, kèm đúng chữ của nó: trong bảng số liệu của bài học (nguon)
    hoặc nguyên văn trong chương (trich)."""
    chu = nguon if nguon is not None else trich
    if match(gia_tri, chu) is None:
        raise ValueError(f"{gia_tri!r} không phải con số ghi trong {chu!r}")
    return gia_tri


def theo_sach(ten, gia_tri, sach, nguon=None, trich=None, tol=None):
    """So một con số tính được với con số của sách. In ✓ khi khớp, ✗ kèm độ lệch khi không."""
    chu = nguon if nguon is not None else trich
    khop = match(sach, chu)
    if khop is None:
        raise ValueError(f"{sach!r} không phải con số ghi trong {chu!r}")
    sai_so = tol if tol is not None else khop[1]
    dung = abs(gia_tri - sach) <= sai_so
    print(f"{'✓' if dung else '✗'} {ten}: tính được {_so(gia_tri)} · sách: {chu}")
    if not dung:
        loi = f"lệch {_so(gia_tri - sach)}, quá sai số cho phép {_so(sai_so)}"
        if NB_STRICT:
            raise AssertionError(f"{ten}: {loi}")
        print(f"   {loi}. Nếu bạn vừa đổi tham số ở trên thì đây là điều được chờ đợi.")
    # None, not the bool: a cell ending in theo_sach(...) would otherwise echo True under the ✓.


def do_tren_may(ten, gia_tri, don_vi=""):
    """Một con số ĐO trên máy đang chạy: mỗi máy một khác, không phải con số của sách."""
    print(f"⏱ {ten}: {_so(gia_tri)} {don_vi} (đo trên máy này)".replace("  ", " "))


print(f"Python {sys.version.split()[0]} · PyTorch {torch.__version__} · NumPy {np.__version__}"
      f" · {torch.get_num_threads()} luồng CPU")

In [ ]:
import pandas as pd
import torch.nn as nn
import torch.nn.functional as F
import torchvision

MAU = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#8a5cc2"]
FP32, FP16, INT8 = 4, 2, 1


def thoi_gian(f, lap=5):
    """Trung vị thời gian của f() (giây), sau một lần chạy khởi động."""
    f()
    do = []
    for _ in range(lap):
        t0 = time.perf_counter()
        f()
        do.append(time.perf_counter() - t0)
    return float(np.median(do))


print("sẵn sàng")

## 1. Chuẩn hóa: theo batch hay theo tầng

Chuẩn hóa theo batch lấy trung bình và phương sai **trên batch**, cho từng kênh. Chuẩn hóa theo tầng lấy chúng
**trên các đặc trưng** của từng mẫu. Một khác biệt về trục, nhưng nó quyết định ba chuyện hệ thống mà chương nêu:
kết quả của một mẫu có phụ thuộc những mẫu khác trong batch không, huấn luyện và suy luận có chạy cùng một phép
tính không, và batch nhỏ thì sao.

**Dự đoán:** nếu bạn thay một mẫu **khác** trong batch, đầu ra của mẫu đầu tiên có đổi không, với mỗi loại?

In [ ]:
dat_hat_giong()
d = 16
bn = nn.BatchNorm1d(d).train()
ln = nn.LayerNorm(d)
x = torch.randn(8, d)
x_doi = x.clone()
x_doi[5] = 10 * torch.randn(d)                    # chỉ thay mẫu số 5
with torch.no_grad():
    lech_bn = (bn(x)[0] - bn(x_doi)[0]).abs().max()
    lech_ln = (ln(x)[0] - ln(x_doi)[0]).abs().max()
print(f"thay mẫu 5, đầu ra của mẫu 0 đổi: chuẩn hóa theo batch {lech_bn:.3f}, chuẩn hóa theo tầng {lech_ln:.1e}")
assert lech_ln == 0

# Định lý 1.3: ma trận Jacobi của phép chuẩn hóa theo batch ghép các mẫu với nhau.
bn1 = nn.BatchNorm1d(1, affine=False).train()
x1 = torch.randn(8)
J = torch.autograd.functional.jacobian(lambda z: bn1(z.view(-1, 1)).view(-1), x1)
ngoai_cheo = (J - torch.diag(torch.diag(J))).abs().max()
print(f"Jacobi 8×8 của chuẩn hóa theo batch: phần tử ngoài đường chéo lớn nhất {ngoai_cheo:.3f} (khác 0: các mẫu ghép nhau)")
print(f"J · (1, 1, ..., 1) = {(J @ torch.ones(8)).abs().max():.1e}: cộng cùng một hằng số vào cả batch thì đầu ra không đổi")
assert ngoai_cheo > 0.01 and (J @ torch.ones(8)).abs().max() < 1e-5

Chiều có trị riêng bằng 0 đó chính là điều Định lý 1.3 nói: phép chuẩn hóa theo batch bất biến khi cả batch dịch
đi cùng một lượng, nên gradient theo hướng đó bằng 0. Không có cận dưới dương chung nào cho các trị riêng.

Giờ đến khác biệt giữa huấn luyện và suy luận. Khi suy luận, chuẩn hóa theo batch không dùng thống kê của batch
nữa mà dùng trung bình chạy (running mean) và phương sai chạy tích luỹ lúc huấn luyện. Quên chuyển chế độ là một
nguồn lệch giữa huấn luyện và phục vụ (training-serving skew) quen thuộc.

In [ ]:
with torch.no_grad():
    lech_che_do_bn = (bn.train()(x) - bn.eval()(x)).abs().max()
    lech_che_do_ln = (ln.train()(x) - ln.eval()(x)).abs().max()
print(f"cùng đầu vào, chế độ huấn luyện so với suy luận: theo batch lệch {lech_che_do_bn:.3f}, theo tầng {lech_che_do_ln:.1e}")
assert lech_che_do_ln == 0

try:
    nn.BatchNorm1d(d).train()(torch.randn(1, d))  # batch 1 khi huấn luyện
except ValueError as loi:
    print("chuẩn hóa theo batch, batch 1, chế độ huấn luyện:", str(loi).split(",")[0])
print("chuẩn hóa theo tầng, batch 1:", tuple(ln(torch.randn(1, d)).shape), "(chạy bình thường)")

D_MO_HINH = 512
rms = nn.RMSNorm(D_MO_HINH, eps=1e-6)
x_r = torch.randn(4, D_MO_HINH)
assert torch.allclose(rms(x_r), x_r / torch.sqrt((x_r**2).mean(-1, keepdim=True) + 1e-6) * rms.weight, atol=1e-6)
for ten, m in (("chuẩn hóa theo batch", nn.BatchNorm1d(D_MO_HINH)), ("chuẩn hóa theo tầng", nn.LayerNorm(D_MO_HINH)),
               ("RMSNorm", rms)):
    dem_dem = sum(b.numel() for b in m.buffers())
    print(f"{ten:>22}, 512 đặc trưng: {sum(p.numel() for p in m.parameters()):>5,} tham số học, "
          f"{dem_dem:>5,} giá trị bộ đệm thống kê")

Chuẩn hóa theo batch mang thêm hai vectơ thống kê phải lưu cùng mô hình và phải dùng đúng chế độ khi phục vụ.
Chuẩn hóa theo tầng không có gì như vậy, và chạy được ở batch 1, nên nó là lựa chọn của transformer, nơi độ dài
chuỗi thay đổi và việc sinh token đi từng mẫu một. RMSNorm bỏ thêm phép trừ trung bình, nên nó cũng bỏ vectơ dịch
$\beta$.

## 2. Kết nối bỏ qua, ngắn gọn

Chương giải thích vì sao mạng sâu khó học bằng một tích $N$ thừa số. Đạo hàm của sigmoid lớn nhất là 0.25, nên qua
$N$ tầng gradient có thể bị nhân với khoảng $0.25^N$; ngược lại, nếu mỗi tầng nở một hướng chung ra khoảng 1.5
lần thì hướng đó tăng theo hàm mũ. Kết nối bỏ qua $y = F(x) + x$ đổi mỗi thừa số thành $I + F'$, và Định lý 1.2 nói:
nếu $\|F'\|_2 = \varepsilon < 1$ thì mọi giá trị kỳ dị của $I + F'$ nằm trong $[1 - \varepsilon,\, 1 + \varepsilon]$.
Ô dưới kiểm cả ba điều bằng phép tính.

In [ ]:
z = torch.zeros(1, requires_grad=True)
torch.sigmoid(z).backward()
theo_sach("đạo hàm của sigmoid tại 0", z.grad.item(), sach=0.25, nguon="0,25 · lũy thừa N")
NO_RA = sach(1.5, nguon="~1,5", tol=0.05)
for N in (sach(10, nguon="10–20 tầng"), sach(20, nguon="10–20 tầng"), sach(56, nguon="20–30 tầng · 56 tầng")):
    print(f"N = {N:>2}: 0.25^N = {0.25**N:.1e},  1.5^N = {NO_RA**N:.1e}")

dat_hat_giong()
for eps in (0.1, 0.3, 0.9):
    A = torch.randn(64, 64)
    A = eps * A / torch.linalg.matrix_norm(A, ord=2)   # ‖F′‖₂ = ε
    s = torch.linalg.svdvals(torch.eye(64) + A)
    assert 1 - eps - 1e-5 <= s.min() and s.max() <= 1 + eps + 1e-5
    print(f"ε = {eps}: giá trị kỳ dị của I + F′ nằm trong [{s.min():.3f}, {s.max():.3f}] ⊂ [{1 - eps:.1f}, {1 + eps:.1f}]")

Cận đó chỉ là cho **một** khối. Qua nhiều khối, các độ lệch vẫn có thể cộng dồn, nên chương nói kết nối bỏ qua
cải thiện điều kiện của bài toán chứ không bảo đảm gradient đi qua nguyên vẹn. Về phía hệ thống, cái giá của kết nối
bỏ qua là đầu vào $x$ phải còn sống trong bộ nhớ cho tới khi hai nhánh gặp lại nhau.

Nếu bạn muốn đi sâu vào hiện tượng tiêu biến gradient (vanishing gradient) và cách sửa nó, có một bài riêng, kèm notebook với các thí
nghiệm chạy được: [Chuyên đề: Tiêu biến gradient trong mạng MLP](https://d3lu8vk4we0dc.cloudfront.net/lessons/topic-vanishing-gradient/)
và [notebook của bài](https://colab.research.google.com/gist/trungnguyenvn/c711c9eb460269492e18e8e4c4e52b5d/vanishing_gradient.ipynb).

## 3. Năng lượng: một phép nhân so với một lần đọc DRAM

Theo số đo của Horowitz mà chương dẫn, một phép nhân FP32 tốn khoảng 3.7 pJ, còn đọc 32 bit từ DRAM tốn 640 pJ.
Chương lưu ý tỉ số đó một mình chưa chứng minh được gì: năng lượng của một tải công việc còn tuỳ số phép tính, số
lần truyền và mức tái sử dụng. Vậy hãy tính cả ba, với một mô hình đơn giản của riêng sổ tay này: mỗi FLOP 3.7 pJ,
và mỗi giá trị 32 bit đi qua DRAM đúng **một** lần (đọc trọng số và đầu vào, ghi đầu ra), mọi thứ khác nằm trên chip.

**Dự đoán:** với tầng dày đặc 784 → 100 ở batch 1, phần nào tốn năng lượng hơn? Với tầng tích chập 3×3, 64 → 64
trên ảnh 224×224 thì sao?

In [ ]:
PJ_FLOP = sach(3.7, nguon="3,7 pJ · 640 pJ")
PJ_DRAM = sach(640, nguon="3,7 pJ · 640 pJ")
theo_sach("đọc DRAM so với một phép nhân (lần)", PJ_DRAM / PJ_FLOP, sach=173, nguon="~173×", tol=0.5)


def nang_luong(flop, gia_tri_dram):
    """(µJ cho phép tính, µJ cho DRAM) theo mô hình ở trên."""
    return flop * PJ_FLOP / 1e6, gia_tri_dram * PJ_DRAM / 1e6


tang = []
for B in (1, 8, 64, 512):                         # tầng dày đặc 784 → 100, batch B
    tang.append((f"dày đặc 784 → 100, batch {B}", *nang_luong(2 * 784 * 100 * B, 784 * 100 + B * (784 + 100)), B))
canh, C = 224, 64                                 # tầng tích chập của sổ tay 01 và 02
tang.append(("tích chập 3×3, 64 → 64, 224×224", *nang_luong(2 * 9 * C * C * canh * canh,
                                                          9 * C * C + 2 * C * canh * canh), 1))
bang_nl = pd.DataFrame(tang, columns=["tầng", "phép tính (µJ)", "DRAM (µJ)", "mẫu"])
bang_nl["DRAM / phép tính"] = bang_nl["DRAM (µJ)"] / bang_nl["phép tính (µJ)"]
bang_nl["tổng mỗi mẫu (µJ)"] = (bang_nl["phép tính (µJ)"] + bang_nl["DRAM (µJ)"]) / bang_nl["mẫu"]
print(bang_nl.drop(columns="mẫu").to_string(index=False, float_format=lambda v: f"{v:,.2f}"))

Ở batch 1, tầng dày đặc tốn cho DRAM nhiều hơn cho phép tính hàng chục lần: mỗi trọng số được đọc về chỉ để dùng
hai phép tính. Gom batch dàn chi phí đọc trọng số cho nhiều mẫu, nên năng lượng mỗi mẫu giảm. Tầng tích chập thì
ngược lại: mỗi trọng số được dùng ở 50,176 vị trí, nên phép tính chiếm phần lớn. Đây là lý do bộ tăng tốc dành
nhiều diện tích cho SRAM trên chip: giữ dữ liệu dùng lại gần đơn vị tính là cách rẻ nhất để không trả 640 pJ lần
nữa.

## 4. RecSys: khi giới hạn là dung lượng

Hệ gợi ý biến mỗi ID người dùng hay sản phẩm thành một vectơ trong bảng embedding (embedding table). Bảng đó lớn
theo số ID, không theo số phép tính. Napkin Math 1.2 của chương tính một bảng 100 triệu sản phẩm, 128 chiều, FP32,
trên thiết bị 80 GB.

In [ ]:
SO_ID = sach(100e6, nguon="100 triệu × 128 × 4 byte")
CHIEU = sach(128, nguon="100 triệu × 128 × 4 byte")
BYTE = sach(4, nguon="100 triệu × 128 × 4 byte")
BO_NHO_THIET_BI = sach(80, nguon="100 triệu ID · 128 FP32 · 80 GB")
bang_gb = SO_ID * CHIEU * BYTE / 1e9
theo_sach("một bảng embedding (GB)", bang_gb, sach=51.2, nguon="51,2 GB · 64 %", tol=0.05)
theo_sach("phần của thiết bị 80 GB (%)", 100 * bang_gb / BO_NHO_THIET_BI, sach=64, nguon="51,2 GB · 64 %")
print(f"thêm một bảng người dùng cùng cỡ: {200 * bang_gb / BO_NHO_THIET_BI:.0f} % dung lượng, không còn vừa một thiết bị")
NGUOI_DUNG = sach(1e9, nguon="1 tỷ người dùng · 128 chiều · 512 GB")
theo_sach("bảng 1 tỷ người dùng (GB)", NGUOI_DUNG * CHIEU * FP32 / 1e9, sach=512,
          nguon="1 tỷ người dùng · 128 chiều · 512 GB")

# Tra bảng là một phép nhân ma trận với vectơ one-hot, nhưng không ai tính nó như vậy.
dat_hat_giong()
bang_nho = torch.randn(1000, 16)
ids = torch.randint(0, 1000, (32,))
assert torch.equal(F.one_hot(ids, 1000).float() @ bang_nho, bang_nho[ids])
print(f"one-hot · bảng = tra bảng (đúng từng bit). Ở {SO_ID:,.0f} ID, cách one-hot tốn {SO_ID * CHIEU:,.0f} MAC "
      f"cho một lần tra; tra trực tiếp chỉ chép {CHIEU} giá trị")

Phép tra bảng không có phép tính nào: nó là một phép chép từ một địa chỉ phụ thuộc dữ liệu. Chương phân biệt hai
giới hạn ở đây. **Tổng cỡ** các bảng làm mô hình nghẽn ở dung lượng; **tốc độ** của từng lần tra làm phép tra nghẽn
ở băng thông, và vì mỗi mẫu chạm vào những hàng khác nhau nên truy cập gần như ngẫu nhiên. Ô dưới đo đúng điều đó:
chép cùng số hàng từ một bảng 200,000 × 128 (102 MB, lớn hơn cache), một lần theo chỉ số ngẫu nhiên, một lần theo
các hàng liền nhau.

In [ ]:
dat_hat_giong()
bang_lon = torch.randn(200_000, CHIEU)
for so_hang in (4096, 65_536):
    ngau_nhien = torch.randint(0, bang_lon.shape[0], (so_hang,))
    lien_nhau = torch.arange(so_hang)
    t_nn = thoi_gian(lambda: bang_lon[ngau_nhien])
    t_ln = thoi_gian(lambda: bang_lon[lien_nhau])
    do_tren_may(f"{so_hang:,} hàng, chỉ số ngẫu nhiên", so_hang * CHIEU * FP32 / t_nn / 1e9, "GB/s")
    do_tren_may(f"{so_hang:,} hàng, liền nhau", so_hang * CHIEU * FP32 / t_ln / 1e9, "GB/s")
NGAN_SACH_MS = sach(50, nguon="dưới 50 ms · hàng triệu mục")
SO_MUC = 1_000_000                                # "hàng triệu mục": lấy một triệu
byte_quet = SO_MUC * CHIEU * FP32
print(f"chấm điểm {SO_MUC:,} mục 128 chiều nghĩa là đọc {byte_quet / 1e6:,.0f} MB; ngân sách là {NGAN_SACH_MS} ms")
do_tren_may("thời gian đọc chừng đó ở băng thông liền nhau vừa đo", 1000 * byte_quet / (so_hang * CHIEU * FP32 / t_ln), "ms")

Hai con số băng thông cho bạn thấy truy cập ngẫu nhiên tốn thêm bao nhiêu trên máy này. Con số cuối là một ước
lượng: nếu nó vượt 50 ms, việc chấm điểm từng mục một không còn khả thi ở quy mô đó, và hệ thống phải lọc ứng viên
trước. Đó là lý do hệ gợi ý thật chia thành nhiều tầng truy xuất.

## 5. Bảng độ phức tạp, tính cho cấu hình cụ thể

Bảng 11 và 12 của chương ghi cách mỗi họ lớn lên bằng ký hiệu $\mathcal{O}$. Ô dưới tính chúng cho một cấu hình cụ
thể mỗi họ, rồi gấp đôi "kích thước đầu vào" của họ đó (số đặc trưng, cạnh ảnh, độ dài chuỗi) và đọc ra hệ số mỗi
đại lượng nhân lên. Cột cuối là số bước phải chạy nối tiếp nhau, tức độ dài đường tới hạn.

In [ ]:
def mlp(n):           # tầng dày đặc n → n
    return dict(tham_so=n * n, kich_hoat=n, mac=n * n, noi_tiep=1)


def cnn(canh):        # tích chập 3×3, 64 → 64, ảnh canh × canh
    return dict(tham_so=9 * 64 * 64, kich_hoat=canh * canh * 64, mac=canh * canh * 9 * 64 * 64, noi_tiep=1)


def rnn(S):           # RNN 100 → 128, chuỗi S bước, khi suy luận
    return dict(tham_so=128 * 128 + 100 * 128, kich_hoat=128, mac=S * (128 * 128 + 100 * 128), noi_tiep=S)


def transformer(S):   # một khối d = 512, 8 đầu, d_ff = 2048, chuỗi S token (giai đoạn xử lý prompt)
    d = 512
    return dict(tham_so=4 * d * d + 2 * d * 2048, kich_hoat=8 * S * S + S * d,
                mac=S * (4 * d * d + 2 * d * 2048) + 2 * S * S * d, noi_tiep=1)


dong = []
for ten, ham, n in (("MLP, 2,048 đặc trưng", mlp, 2048), ("CNN, ảnh 224×224", cnn, 224),
                    ("RNN, 1,000 bước", rnn, 1000), ("transformer, 1,024 token", transformer, 1024)):
    a, b = ham(n), ham(2 * n)
    dong.append([ten, a["tham_so"], a["kich_hoat"], a["mac"], a["noi_tiep"]]
                + [b[k] / a[k] for k in ("tham_so", "kich_hoat", "mac", "noi_tiep")])
bang_pt = pd.DataFrame(dong, columns=["cấu hình", "tham số", "trạng thái/kích hoạt", "MAC", "bước nối tiếp",
                                      "×tham số", "×kích hoạt", "×MAC", "×nối tiếp"])
print("khi kích thước đầu vào gấp đôi:")
print(bang_pt.to_string(index=False, formatters={c: "{:,.0f}".format for c in bang_pt.columns[1:5]}
                        | {c: "{:g}".format for c in bang_pt.columns[5:]}))

Mỗi họ có một cột lớn lên nhanh nhất, và đó chính là điểm nghẽn mà chương gán cho nó. Tầng dày đặc gấp bốn số tham
số. Tích chập giữ nguyên tham số nhưng gấp bốn giá trị kích hoạt và phép tính. RNN giữ nguyên trạng thái nhưng gấp đôi
đường tới hạn. Transformer gần gấp bốn bộ nhớ điểm (ở đây là 8 đầu, chưa chia khối). Cột "trạng thái/kích hoạt" của
RNN là trạng thái lúc suy luận; khi huấn luyện, nó cũng lớn theo độ dài chuỗi.

## 6. Ba chữ ký trong định luật sắt

Phần tổng kết của chương viết định luật sắt ở dạng nối tiếp:

$$T = \frac{D_{vol}}{BW} + \frac{O}{R_{peak} \cdot \eta_{hw}} + L_{lat}.$$

Mỗi chữ ký tải công việc làm một số hạng lớn nhất. Ô dưới đặt ba tải công việc lên **một thiết bị giả định**:
thông lượng đỉnh 10 TFLOP/s và mức sử dụng 50 %, lấy từ Napkin Math 1.3, còn băng thông 0.5 TB/s và phụ trội 5 µs
cho mỗi bước nối tiếp là con số tự chọn của sổ tay này, không phải của sách. Rồi nó hỏi câu mà bài lab của chương hỏi:
gấp đôi sức tính hay gấp đôi băng thông thì giúp tải công việc nào?

In [ ]:
R_DINH = sach(10, nguon="10 TFLOP/s · 50–60 %") * 1e12
ETA = sach(50, nguon="10 TFLOP/s · 50–60 %") / 100
BW_GIA_DINH = 0.5e12                               # giả định
PHU_TROI_BUOC = 5e-6                               # giả định, mỗi bước nối tiếp

BUOC = sach(1000, trich="the system must execute 1,000 dependent matrix-vector multiplications")
W_RNN = 128 * 128 + 100 * 128                      # RNN 100 → 128 của sổ tay 03; trọng số nằm trên chip suốt chuỗi
tai = {   # (O FLOP, D byte, số bước nối tiếp chịu phụ trội)
    "ResNet-50, một ảnh": (sach(8.2, nguon="25,6M · 8,2 GFLOP") * 1e9, sach(102.4, nguon="25,6M · 102,4 MB") * 1e6, 0),
    "GPT-2 XL, một token": (sach(3, nguon="6 GB · 3 GFLOP/token") * 1e9, sach(6, nguon="6 GB · 3 GFLOP/token") * 1e9, 0),
    "RNN 100 → 128, 1,000 bước": (2 * BUOC * W_RNN, W_RNN * FP32, BUOC),
}


def dinh_luat_sat(O, D, buoc, R=R_DINH, bw=BW_GIA_DINH):
    return {"D/BW": D / bw, "O/(R·η)": O / (R * ETA), "L_lat": buoc * PHU_TROI_BUOC}


dong = []
for ten, (O, D, buoc) in tai.items():
    goc = dinh_luat_sat(O, D, buoc)
    T = sum(goc.values())
    dong.append([ten, *(1000 * v for v in goc.values()), 1000 * T, max(goc, key=goc.get),
                 T / sum(dinh_luat_sat(O, D, buoc, R=2 * R_DINH).values()),
                 T / sum(dinh_luat_sat(O, D, buoc, bw=2 * BW_GIA_DINH).values())])
bang_sat = pd.DataFrame(dong, columns=["tải công việc", "D/BW (ms)", "O/(R·η) (ms)", "L_lat (ms)", "T (ms)",
                                       "số hạng lớn nhất", "nhanh hơn khi gấp đôi R", "nhanh hơn khi gấp đôi BW"])
print(bang_sat.to_string(index=False, float_format=lambda v: f"{v:,.3g}"))

fig, ax = plt.subplots(figsize=(7.6, 3.2))
trai = np.zeros(len(bang_sat))
for cot, mau in zip(("D/BW (ms)", "O/(R·η) (ms)", "L_lat (ms)"), (MAU[0], MAU[1], MAU[4])):
    ax.barh(bang_sat["tải công việc"], bang_sat[cot], left=trai, color=mau, height=0.5, label=cot.replace(" (ms)", ""))
    trai += bang_sat[cot].to_numpy()
ax.invert_yaxis()
ax.set_xlim(0, 1.1 * bang_sat["T (ms)"].max())
ax.set_xlabel("thời gian (ms) trên thiết bị giả định")
ax.set_title("Ba số hạng của định luật sắt cho ba chữ ký tải công việc")
ax.legend(loc="lower right", fontsize=8)
plt.tight_layout()
plt.show()

Ba tải công việc trả lời ba cách khác nhau: ResNet-50 nhanh lên khi gấp đôi sức tính, GPT-2 XL khi gấp đôi băng
thông, còn RNN gần như không nhanh lên với cả hai, vì thời gian của nó nằm ở chuỗi bước nối tiếp. Bảng này phụ thuộc
thiết bị giả định: ở thiết bị có băng thông thấp hơn so với sức tính, ResNet-50 batch 1 cũng có thể chuyển sang nghẽn
ở bộ nhớ, và đó là lúc batch kéo nó về. Điều không phụ thuộc thiết bị là hình dạng câu trả lời: muốn biết nâng cấp
nào có ích, trước hết phải biết số hạng nào lớn nhất.

Napkin Math 1.3 dùng cùng thiết bị để hỏi một câu khác: chạy ResNet-50 ở 30 khung hình mỗi giây thì còn bao nhiêu dư
địa?

In [ ]:
FPS = sach(30, nguon="30 FPS · 8,2 GFLOP · 246 GFLOP/s")
can = FPS * sach(8.2, nguon="30 FPS · 8,2 GFLOP") * 1e9
theo_sach("thông lượng bền cần có (GFLOP/s)", can / 1e9, sach=246, nguon="30 FPS · 8,2 GFLOP · 246 GFLOP/s")
hieu_dung = R_DINH * ETA
theo_sach("thông lượng hiệu dụng ở 50 % (TFLOP/s)", hieu_dung / 1e12, sach=5, nguon="50–60 % · 5 TFLOP/s")
theo_sach("dư địa cho ResNet-50 (lần)", hieu_dung / can, sach=20.3, nguon="5 TFLOP/s · 20,3×")
phat_hien = FPS * sach(100, nguon="100 GFLOP · 3 TFLOP/s · 1,7×") * 1e9
theo_sach("mô hình phát hiện vật thể ở 30 FPS (TFLOP/s)", phat_hien / 1e12, sach=3, nguon="100 GFLOP · 3 TFLOP/s")
theo_sach("dư địa còn lại (lần)", hieu_dung / phat_hien, sach=1.7, nguon="3 TFLOP/s · 1,7×")

## 7. Chọn kiến trúc cho máy bẫy ảnh

Ví dụ cuối của chương là một máy bẫy ảnh trong vườn quốc gia: không có mạng, chạy pin sáu tháng, nhận 50 loài với độ
chính xác 90 %, ảnh 1920×1080 thu về 224×224. Thiết bị là một ARM Cortex-A53 khoảng 2 GOPS INT8 với 512 MB RAM. Bài
lab của chương làm đúng việc này: mỗi yêu cầu là một phép thử đạt hoặc không, một ứng viên khả thi khi đạt **mọi**
phép thử, rồi mới chọn theo ưu tiên trong số còn lại.

Ứng viên được chọn của chương là MobileNetV2 với hệ số độ rộng 0.75. Torchvision dựng được nó, nên ta đếm thẳng.

In [ ]:
DO_RONG = sach(0.75, nguon="1920×1080 → 224×224 · 0,75")
dat_hat_giong()
mnv2 = torchvision.models.mobilenet_v2(weights=None, width_mult=DO_RONG)
mac_mnv2 = []
moc = [m.register_forward_hook(lambda m, i, o: mac_mnv2.append(
    o.numel() * (m.in_features if isinstance(m, nn.Linear)
                 else m.kernel_size[0] * m.kernel_size[1] * m.in_channels // m.groups)))
    for m in mnv2.modules() if isinstance(m, (nn.Conv2d, nn.Linear))]
with torch.no_grad():
    mnv2.eval()(torch.zeros(1, 3, 224, 224))
for h in moc:
    h.remove()
tham_so_mnv2 = sum(p.numel() for p in mnv2.parameters())
theo_sach("tham số MobileNetV2 0.75", tham_so_mnv2, sach=2.6e6, trich="It carries ~2.6M parameters", tol=5e4)
theo_sach("MFLOP mỗi ảnh 224×224", 2 * sum(mac_mnv2) / 1e6, sach=418, trich="~418 MFLOP", tol=1)
theo_sach("MB trọng số FP32, từ con số đã làm tròn 2.6 triệu", 2.6e6 * FP32 / 1e6, sach=10.4,
          trich="(10.4 MB FP32, 2.6 MB with INT8 weights)")
print(f"với số tham số chính xác {tham_so_mnv2:,}: {tham_so_mnv2 * FP32 / 1e6:.2f} MB FP32, "
      f"{tham_so_mnv2 * INT8 / 1e6:.2f} MB INT8")

Giờ là bảng khả thi. Mỗi hàng là một ứng viên của chương, với số tham số và FLOP chương ghi. Ba phép thử tính được:

* **trọng số** ở INT8 nằm trong 100 MB mà chương dành cho trọng số;
* **độ trễ** = số phép tính chia cho 2 GOPS, như chương làm cho MobileNetV2, phải dưới 500 ms;
* **năng lượng** mỗi lần suy luận ở 200 mW (con số chương ước cho MobileNetV2, ở đây áp chung cho mọi ứng viên, một
  giả định của sổ tay này).

Phép thử thứ tư, đủ năng lực cho 50 loài, không tính được từ những con số này; cột đó chép đánh giá của chương.

In [ ]:
GOPS = sach(2, trich="~2 GOPS INT8", tol=0) * 1e9
TRE_MUC_TIEU = sach(500, nguon="~2 W · 209 ms · 500 ms", tol=0)
TRONG_SO_MB = sach(100, trich="the 100 MB this deployment allocates to model weights")
CONG_SUAT_W = sach(200, trich="~200 mW for 209 ms, or 41.8 mJ per inference", tol=0) / 1000
RAM_MB = sach(512, trich="512 MB RAM, 2 GB storage")

ung_vien = [  # tên, tham số, FLOP mỗi ảnh, đủ năng lực theo chương
    ("ResNet-50", sach(25.6e6, nguon="ResNet-50: 25,6M · 102,4 MB"), sach(8.2, nguon="25,6M · 8,2 GFLOP") * 1e9, True),
    ("MobileNetV1", sach(4.2, trich="MobileNetV1 (4.2M params, 1138 MFLOP)") * 1e6,
     sach(1138, trich="MobileNetV1 (4.2M params, 1138 MFLOP)") * 1e6, True),
    ("MobileNetV2 0.75", sach(2.6e6, trich="It carries ~2.6M parameters", tol=5e4),
     sach(418, trich="~418 MFLOP", tol=1) * 1e6, True),
    ("KWS DS-CNN", sach(200, trich="KWS DS-CNN (200K params, 20 MFLOP)") * 1e3,
     sach(20, trich="KWS DS-CNN (200K params, 20 MFLOP)") * 1e6, False),
]
dong = []
for ten, p, flop, du_nang_luc in ung_vien:
    tre_ms = 1000 * flop / GOPS
    kq = {"trọng số INT8 (MB)": p * INT8 / 1e6, "độ trễ (ms)": tre_ms, "năng lượng (mJ)": CONG_SUAT_W * tre_ms}
    dat = {"trọng số": kq["trọng số INT8 (MB)"] <= TRONG_SO_MB, "độ trễ": tre_ms <= TRE_MUC_TIEU,
           "năng lực (theo chương)": du_nang_luc}
    dong.append({"ứng viên": ten, **kq, **{f"đạt {k}": "có" if v else "không" for k, v in dat.items()},
                 "khả thi": all(dat.values())})
bang_kt = pd.DataFrame(dong)
print(bang_kt.to_string(index=False, float_format=lambda v: f"{v:,.1f}"))
kha_thi = bang_kt[bang_kt["khả thi"]]
chon = kha_thi.sort_values("độ trễ (ms)").iloc[0]
print(f"\ncòn lại sau mọi phép thử: {', '.join(kha_thi['ứng viên'])}; ưu tiên độ trễ thấp nhất: {chon['ứng viên']}")

theo_sach("độ trễ của MobileNetV2 0.75 (ms)", chon["độ trễ (ms)"], sach=209, nguon="~2 W · 209 ms · 500 ms", tol=0.5)
theo_sach("năng lượng mỗi lần suy luận (mJ)", chon["năng lượng (mJ)"], sach=41.8,
          trich="~200 mW for 209 ms, or 41.8 mJ per inference", tol=0.05)
LAN_MOI_NGAY = sach(100, trich="At 100 inferences/day")
theo_sach("năng lượng mỗi ngày cho suy luận (J)", chon["năng lượng (mJ)"] * LAN_MOI_NGAY / 1000, sach=4.2,
          trich="4.2 J/day")
print(f"{TRONG_SO_MB} MB cho trọng số là {100 * TRONG_SO_MB / RAM_MB:.0f} % của {RAM_MB} MB RAM; "
      f"MobileNetV2 0.75 ở INT8 dùng {100 * chon['trọng số INT8 (MB)'] / TRONG_SO_MB:.1f} % phần đó")

Bảng loại ResNet-50 vì độ trễ (4,100 ms ở 2 GOPS) chứ không vì trọng số, đúng như chương nói: chặn nó là số phép tính
và điện năng, không phải dung lượng lưu trữ. KWS bị loại vì năng lực, phép thử duy nhất không tính được. Có một điều
chương không tính: với cùng giả định 2 GOPS, MobileNetV1 cần 569 ms, vượt mục tiêu 500 ms. Chương gọi nó là "hứa
hẹn" rồi chuyển sang MobileNetV2 vì cùng việc mà ít tài nguyên hơn; phép chia này cho thêm một lý do cụ thể.

Năng lượng mỗi ngày ở đây chỉ là phần suy luận. Chương nhắc rằng cảm biến, chế độ ngủ, lưu trữ và truyền thông còn
chưa tính, nên mục tiêu sáu tháng pin vẫn phải kiểm trên thiết bị thật.

### Khoảng cách triển khai

Ngộ nhận cuối của chương: một mô hình huấn luyện trên tám A100 có thể chạy trên một Jetson Orin NX.

In [ ]:
SO_GPU = sach(8, nguon="8× A100 · 640 GB")
MOT_A100 = sach(80, trich="Those values do not exhaust a 80 GB device")
theo_sach("bộ nhớ của tám A100 (GB)", SO_GPU * MOT_A100, sach=640, nguon="8× A100 · 640 GB")
JETSON = sach(16, nguon="640 GB → Jetson Orin NX 16 GB")
print(f"khoảng cách: {SO_GPU * MOT_A100 / JETSON:.0f} lần bộ nhớ")
for kieu, byte in (("FP16", FP16), ("INT8", INT8)):
    print(f"  {JETSON} GB chứa tối đa {JETSON / byte:.0f} tỷ tham số ở {kieu}, khi chưa tính bộ nhớ đệm KV, "
          f"giá trị kích hoạt hay hệ điều hành")
TY_70 = sach(70, trich="a 70-billion-parameter model stores 140 GB of FP16 weights")
print(f"mô hình 70 tỷ tham số cần {TY_70 * FP16:.0f} GB ở FP16: gấp {TY_70 * FP16 / JETSON:.1f} lần cả bộ nhớ Jetson")

Bốn mươi lần bộ nhớ không phải khoảng cách mà một phép tối ưu nhỏ lấp được. Muốn đi từ cụm huấn luyện xuống thiết bị
biên, kiến trúc phải được chọn cho thiết bị biên ngay từ đầu, hoặc mô hình phải được nén theo những cách mà chương Nén
mô hình trình bày.

## Thử thêm

1. Ở phần 6, đặt `BW_GIA_DINH = 0.1e12`. Số hạng lớn nhất của ResNet-50 có đổi không? Rồi chia `D` của nó cho 8 (giả
   sử batch 8 dùng chung một lần đọc trọng số, và tính `O` cho 8 ảnh) để xem batch kéo nó về đâu.
2. Ở phần 7, thêm một ứng viên `("MobileNetV2 1.0", 3.5e6, 600e6, True)` (số của sổ tay 01) và một mục tiêu độ trễ
   250 ms. Ứng viên nào còn lại?
3. Ở phần 1, đặt batch 2 cho chuẩn hóa theo batch (`x[:2]`) và so lệch giữa chế độ huấn luyện và suy luận với batch 8.

## Tóm lại

* Chuẩn hóa theo batch ghép các mẫu trong batch và chạy khác nhau khi huấn luyện và suy luận; chuẩn hóa theo tầng xử
  lý từng mẫu riêng, nên nó hợp với transformer và với batch 1.
* Một khối có kết nối bỏ qua có Jacobi gần ma trận đơn vị khi nhánh dư nhỏ, nhưng điều đó không bảo đảm gì qua mọi độ
  sâu.
* Ở batch 1, tầng dày đặc tốn năng lượng cho DRAM nhiều hơn cho phép tính hàng chục lần; tích chập thì ngược lại vì
  nó dùng lại trọng số.
* Một bảng embedding 100 triệu hàng đã chiếm 64 % một thiết bị 80 GB: RecSys nghẽn ở dung lượng trước khi nghẽn ở
  phép tính.
* Chọn kiến trúc là một chuỗi phép thử đạt hoặc không trên con số của thiết bị đích, và số hạng lớn nhất của định luật
  sắt cho biết nâng cấp nào có ích.

---
### Nguồn và giấy phép

Sổ tay này đi kèm bài học [Kiến trúc mạng](https://d3lu8vk4we0dc.cloudfront.net/lessons/vol1-ch06-nn-architectures/), dựng từ chương
[*Network Architectures*](https://mlsysbook.ai/vol1/nn_architectures/nn_architectures.html) của sách. Một phần sổ tay phỏng theo phòng thí nghiệm `labs/vol1/lab_06_nn_arch.py` của sách (© 2026 President and Fellows of Harvard College, CC BY-NC-SA 4.0).

Nội dung gốc thuộc Machine Learning Systems của Vijay Janapa Reddi và cộng sự, MIT Press. Bản quyền © 2024-2026 Harvard University. Giấy phép CC BY-NC-SA 4.0. Bản tiếng Việt này là tác phẩm phái sinh dùng cùng giấy phép: ghi công, chia sẻ tương tự, phi thương mại.